### Double-check the entries in your final UpperSco table cf. the literature.

In [1]:
import re
import numpy  as np
import pandas as pd
pd.set_option('display.max_columns', None)

`1:` Load Table 1 by Carpenter et al. (2025) and check that the shared columns match your extended table.

In [2]:
# Load Carpenter2025's table:
carpenter = pd.read_csv(
    "table1_carpenter2025.dat",
    names  = [
        "SourceID_2MASS_L22", "SourceID_GaiaDR3_L22", "SpT", "DiscClass_L22E18", "Distance (pc)", "e_d_upper (pc)", "e_d_lower (pc)",
        "KinematicMemebership", "MemberC25",
    ],
    sep      = r"\s+",
    skiprows = 30
)
print(len(carpenter))
# carpenter
# Numbers match: 284.

284


In [3]:
# Load your extended version of the table:
uppersco_mine = pd.read_csv(
    "uppersco_vs_polnitzky.dat",
    names  = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "discClassC25", "discClassL22", "COsizeS21?", "comments"
    ],
    sep      = r"\s+",
    skiprows = 1
)
print(len(uppersco_mine))
# Numbers match: 284.

284


In [4]:
# Compare entries:
# Check that they have the same 2MASS IDs -> they do!
# mask = carpenter["SourceID_2MASS"] != uppersco_mine["SourceID_2MASS"]
# print(uppersco_mine.loc[mask, ["SourceID_2MASS", "SourceID_2MASS"]])
# Check that they have the same GaiaDR3 IDs -> they do!
# mask = carpenter["SourceID_GaiaDR3"] != uppersco_mine["SourceID_GaiaDR3"]
# print(uppersco_mine.loc[mask, ["SourceID_GaiaDR3", "SourceID_GaiaDR3"]])
# Check that they have the same SpTs -> they do!
# for a, b in zip(carpenter["SpT"], uppersco_mine["SpT_L22"]):
#     if a != b: print(f"{a}, {b}")
# Check that they have the kinematic membership -> they do!
# for a, b in zip(carpenter["Kinematic memebership"], uppersco_mine["kinpopL22"]):
#     if a != b: print(f"{a}, {b}")
# for i in range(len(carpenter["SourceID_2MASS"])):
#     if carpenter["MemberC25"][i] != uppersco_mine["memK_L22"][i]: 
#         print(carpenter["SourceID_2MASS"][i], carpenter["MemberC25"][i], uppersco_mine["memK_L22"][i])
# Check that they have the same disc classification -> they do!
# for i in range(len(carpenter["SourceID_2MASS"])):
#     if carpenter["DiscClass"][i] != uppersco_mine["discClassC25"][i]: 
#         print(carpenter["SourceID_2MASS"][i], carpenter["DiscClass"][i], uppersco_mine["discClassC25"][i])
# Check that that the Carpenter and Luhman classification match (on members) -> they do!
# for i in range(len(uppersco_mine["SourceID_2MASS"])):
#     if uppersco_mine["discClassL22"][i] != uppersco_mine["discClassC25"][i]: 
#         print(uppersco_mine["SourceID_2MASS"][i], uppersco_mine["kinpopL22"][i], uppersco_mine["discClassC25"][i], uppersco_mine["discClassL22"][i])

`2:` Load Luhman2022's tables (stars and discs) and cross-check that what Carpenter and you added is correct.
- In stars, check: "SourceID_2MASS", "SourceID_WISE", "SourceID_GaiaDR3", "SpT_L22", "kinpopL22", "pospopL22";
- In discs, check: "SourceID_2MASS", "SourceID_WISE", "discClassL22".

In [5]:
# Load Luhman22a's table:
luhman_stars = pd.read_fwf(
    "table1_luhman2022a_orig.dat",
    names  = [
        "SourceID_GaiaDR3", "SourceID_GaiaDR2", "NameOther",
        "RAdeg", "DEdeg",
        "SpType", "r_SpType", "AdoptSpType",
        "EWLi", "r_EWLi",
        "pmRA", "e_pmRA", "pmDec", "e_pmDec", "plx", "e_plx",
        "r_med_geo", "r_lo_geo", "r_hi_geo",
        "RVel", "e_RVel", "r_RVel",
        "U U", "e_U", "V V", "e_V", "W W", "e_W",
        "Gmag G", "e_Gmag", "GBPmag", "e_GBPmag", "GRPmag", "e_GRPmag",
        "RUWE", "kin", "pos",
        "2MASS", "SourceID_2MASS_L22", "2MASS_angsep", "2MASS_closest",
        "WISE", "SourceID_WISE_L22", "WISE_angsep", "WISE_closest"
    ],
    skiprows = 273,
)
print(len(luhman_stars))
# Numbers match: 10509, yes!

10509


In [6]:
# Load Luhman22b's table:
luhman_discs = pd.read_fwf(
    "table1_luhman2022b_orig.dat",
    names  = [
        "WISE", "SourceID_WISE_L22", "RAdeg","DEdeg", "2MASS", "SourceID_2MASS", "2MASS_angsep",
        "Jmag 2MASS", "e_Jmag", "Hmag 2MASS", "e_Hmag", "Ksmag 2MASS", "e_Ksmag",
        "W1mag", "e_W1mag", "W2mag", "e_W2mag", "W3mag", "e_W3mag", "f_W3mag", "W4mag", "e_W4mag", "f_W4mag",
        "ExcW2 Excess present in W2?", "ExcW3 Excess present in W3?", "ExcW4 Excess present in W4?", "DiskType Disk Typec"
    ],
    skiprows = 45,
)
print(len(luhman_discs))
# Numbers match: 9675, yes!

9675


In [7]:
# Using your 2MASS ID match the number of stars with disc in Luhman (2022a).
# There are some duplicates (sources with same 2MASS ID but different Gaia ID, see below). Trim by GaiaDR3 ID and check 2MASS and WISE:
matched_stars = uppersco_mine.merge(
    luhman_stars[["SourceID_GaiaDR3", "SourceID_2MASS_L22", "2MASS_closest", "SourceID_WISE_L22", "WISE_closest", "AdoptSpType", "kin", "pos"]],
    on  = "SourceID_GaiaDR3",
    how = "left"
)
print(len(matched_stars))

# Using your 2MASS ID match the number of stars with disc in Luhman (2022b):
matched_discs = uppersco_mine.merge(
    luhman_discs[["SourceID_2MASS", "SourceID_WISE_L22", "DiskType Disk Typec"]],
    on  = "SourceID_2MASS",
    how = "left"
)
print(len(matched_discs))

284
284


In [8]:
# Check that the 2MASS ID in my table and Luhman match:
# Some Gaia sources in my table have a 2MASS ID that is absent from Luhman's table, but they are not members (data back to the classification of Esplin et al., 2018)
for i in range(len(matched_stars["SourceID_GaiaDR3"])):
    if matched_stars["SourceID_2MASS"][i] != matched_stars["SourceID_2MASS_L22"][i]: 
        print(
            matched_stars["SourceID_GaiaDR3"][i],
            matched_stars["SourceID_2MASS"][i], matched_stars["SourceID_2MASS_L22"][i], matched_stars["2MASS_closest"][i], 
            matched_stars["memK_L22"][i]
        )

6241506333509857664 J15442550-2126408 nan nan -
6233762782358231808 J15520884-2723456 nan nan -
6246807830684399488 J15534211-2049282 nan nan -
6236547707871265408 J15554314-2437435 nan nan -
6237356707903924608 J15572986-2258438 nan nan -
6237190612936085120 J15583692-2257153 nan nan -
6040286359786429568 J15594231-2945495 nan nan -
6243457550035199744 J15594460-2155250 nan nan -
6248378556063992576 J15595759-1812234 nan nan -
6243217104889830912 J16001844-2230114 nan nan -
6235879479683450496 J16011398-2516281 nan nan -
6243221193699945984 J16020287-2236139 nan nan -
6236047430087176576 J16041893-2430392 nan nan -
6244202748339502848 J16052661-1957050 nan nan -
6247245990371393408 J16062196-1928445 nan nan -
6244217076350451968 J16082751-1949047 nan nan -
6242475862249989888 J16093164-2229224 nan nan -
6243943882070642176 J16093730-2027250 nan nan -
6245780856763065984 J16102857-1904469 nan nan -
6042662198252863488 J16112601-2631558 nan nan -
6243868702963748352 J16120505-2043404 na

In [9]:
# Same for WISE:
for i in range(len(matched_stars["SourceID_GaiaDR3"])):
    if matched_stars["SourceID_WISE"][i] != matched_stars["SourceID_WISE_L22"][i]: 
        print(
            matched_stars["SourceID_GaiaDR3"][i],
            matched_stars["SourceID_WISE"][i],matched_stars["SourceID_WISE_L22"][i], matched_stars["WISE_closest"][i], 
            matched_stars["memK_L22"][i]
        )

6241506333509857664 -99 nan nan -
6233762782358231808 -99 nan nan -
6246807830684399488 -99 nan nan -
6236547707871265408 -99 nan nan -
6237356707903924608 -99 nan nan -
6237190612936085120 -99 nan nan -
6040286359786429568 -99 nan nan -
6243457550035199744 -99 nan nan -
6248378556063992576 -99 nan nan -
6243217104889830912 -99 nan nan -
6235879479683450496 -99 nan nan -
6243221193699945984 -99 nan nan -
6236047430087176576 -99 nan nan -
6244202748339502848 -99 nan nan -
6247245990371393408 -99 nan nan -
6244217076350451968 -99 nan nan -
6242475862249989888 -99 nan nan -
6243943882070642176 -99 nan nan -
6245780856763065984 -99 nan nan -
6042662198252863488 -99 nan nan -
6243868702963748352 -99 nan nan -
6049478414428226688 -99 nan nan -
6049575721207795584 -99 nan nan -
6050394479417395456 -99 nan nan -
6050407948434871808 -99 nan nan -
6050405234012889856 -99 nan nan -
6245413035762118528 -99 nan nan -
6245617643707106432 -99 nan nan -
6038474914382457344 -99 nan nan -
62447521261946

In [10]:
# Compare SpTs (some differences updated following Luhman (2022a); missing SpTs for non members dating back to the classification of Esplin et al. (2018):
for i in range(len(matched_stars["SourceID_2MASS"])):
    if matched_stars["SpT_L22"][i] != matched_stars["AdoptSpType"][i]: 
        print(
            matched_stars["SourceID_2MASS"][i], 
            matched_stars["AdoptSpType"][i], matched_stars["SpT_L22"][i], matched_stars["kin"][i]
        )

J15442550-2126408 nan M3.25 nan
J15462696-2443225 M4.7 M4.75 c
J15472572-2609185 M4.7 M4.75 c
J15482445-2235495 M3.7 M3.75 uc
J15485435-2443101 M4.7 M4.75 uc
J15510126-2523100 M4.7 M4.75 lc
J15520884-2723456 nan M4 nan
J15534211-2049282 nan M3.4 nan
J15554314-2437435 nan M5 nan
J15565600-2911349 M4.7 M4.75 c
J15571045-2545307 M4.7 M4.75 uc
J15572986-2258438 nan M4 nan
J15573049-1903014 M3.2 M3.25 u
J15583620-1946135 M4.2 M4.25 ul
J15583692-2257153 nan K0 nan
J15594231-2945495 nan M3.5 nan
J15594460-2155250 nan M4.75 nan
J15595116-2311044 M2.2 M2.25 u
J15595759-1812234 nan M3.25 nan
J16001844-2230114 nan M4.5 nan
J16011398-2516281 nan M4.5 nan
J16012268-2408003 M2.2 M2.25 u
J16020287-2236139 nan - nan
J16023587-2320170 M4.2 M4.25 u
J16024152-2138245 M4.7 M4.75 u
J16025855-2256495 M2.7 M2.75 u
J16030161-2207523 M4.7 M4.75 u
J16031329-2112569 M4.7 M4.75 u
J16041893-2430392 nan M3 nan
J16043916-1942459 M3.2 M3.25 u
J16050647-1734020 M4.7 M4.75 u
J16052661-1957050 nan M4.5 nan
J16052730-261

In [11]:
# Compare the kinematic populations (no differences):
for i in range(len(matched_stars["SourceID_2MASS"])):
    if matched_stars["kinpopL22"][i] != matched_stars["kin"][i]: 
        print(matched_stars["SourceID_2MASS"][i], matched_stars["kinpopL22"][i], matched_stars["kin"][i])

J15442550-2126408 - nan
J15520884-2723456 - nan
J15534211-2049282 - nan
J15554314-2437435 - nan
J15572986-2258438 - nan
J15583692-2257153 - nan
J15594231-2945495 - nan
J15594460-2155250 - nan
J15595759-1812234 - nan
J16001844-2230114 - nan
J16011398-2516281 - nan
J16020287-2236139 - nan
J16041893-2430392 - nan
J16052661-1957050 - nan
J16062196-1928445 - nan
J16082751-1949047 - nan
J16093164-2229224 - nan
J16093730-2027250 - nan
J16102857-1904469 - nan
J16112601-2631558 - nan
J16120505-2043404 - nan
J16123352-2543281 - nan
J16133650-2503473 - nan
J16135434-2320342 - nan
J16141107-2305362 - nan
J16145131-2308515 - nan
J16153220-2010236 - nan
J16154416-1921171 - nan
J16160448-2932400 - nan
J16165083-2009081 - nan
J16171889-2230017 - nan
J16181618-2619080 - nan
J16181811-2221150 - nan
J16185382-2053182 - nan
J16192288-2135037 - nan
J16202291-2227041 - nan
J16203960-2634284 - nan
J16204468-2431384 - nan
J16215466-2043091 - nan
J16215741-2238180 - nan
J16222982-2002472 - nan
J16235468-251539

In [12]:
# Compare the kinematic populations (no differences):
for i in range(len(matched_stars["SourceID_2MASS"])):
    if matched_stars["pospopL22"][i] != matched_stars["pos"][i]: 
        print(matched_stars["SourceID_2MASS"][i], matched_stars["pospopL22"][i], matched_stars["pos"][i])

J15442550-2126408 - nan
J15520884-2723456 - nan
J15534211-2049282 - nan
J15554314-2437435 - nan
J15572986-2258438 - nan
J15583692-2257153 - nan
J15594231-2945495 - nan
J15594460-2155250 - nan
J15595759-1812234 - nan
J16001844-2230114 - nan
J16011398-2516281 - nan
J16020287-2236139 - nan
J16041893-2430392 - nan
J16052661-1957050 - nan
J16062196-1928445 - nan
J16082751-1949047 - nan
J16093164-2229224 - nan
J16093730-2027250 - nan
J16102857-1904469 - nan
J16112601-2631558 - nan
J16120505-2043404 - nan
J16123352-2543281 - nan
J16133650-2503473 - nan
J16135434-2320342 - nan
J16141107-2305362 - nan
J16145131-2308515 - nan
J16153220-2010236 - nan
J16154416-1921171 - nan
J16160448-2932400 - nan
J16165083-2009081 - nan
J16171889-2230017 - nan
J16181618-2619080 - nan
J16181811-2221150 - nan
J16185382-2053182 - nan
J16192288-2135037 - nan
J16202291-2227041 - nan
J16203960-2634284 - nan
J16204468-2431384 - nan
J16215466-2043091 - nan
J16215741-2238180 - nan
J16222982-2002472 - nan
J16235468-251539

In [13]:
# Check that the WISE IDs are the same:
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["SourceID_WISE"][i] != matched_discs["SourceID_WISE_L22"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["SourceID_WISE"][i], matched_discs["SourceID_WISE_L22"][i])

J15442550-2126408 -99 nan
J15520884-2723456 -99 nan
J15534211-2049282 -99 nan
J15554314-2437435 -99 nan
J15572986-2258438 -99 nan
J15583692-2257153 -99 nan
J15594231-2945495 -99 nan
J15594460-2155250 -99 nan
J15595759-1812234 -99 nan
J16001844-2230114 -99 nan
J16011398-2516281 -99 nan
J16020287-2236139 -99 nan
J16041893-2430392 -99 nan
J16052661-1957050 -99 nan
J16062196-1928445 -99 nan
J16082751-1949047 -99 nan
J16093164-2229224 -99 nan
J16093730-2027250 -99 nan
J16102857-1904469 -99 nan
J16104202-2101319 J161041.99-210132.0 nan
J16112601-2631558 -99 nan
J16120505-2043404 -99 nan
J16123352-2543281 -99 nan
J16133650-2503473 -99 nan
J16135434-2320342 -99 nan
J16141107-2305362 -99 nan
J16145131-2308515 -99 nan
J16153220-2010236 -99 nan
J16154416-1921171 -99 nan
J16160448-2932400 -99 nan
J16165083-2009081 -99 nan
J16171889-2230017 -99 nan
J16181618-2619080 -99 nan
J16181811-2221150 -99 nan
J16185382-2053182 -99 nan
J16192288-2135037 -99 nan
J16202291-2227041 -99 nan
J16203960-2634284 -99 

In [14]:
# Check that the disc classification is the same:
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["discClassL22"][i] != matched_discs["DiskType Disk Typec"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["DiskType Disk Typec"][i], matched_discs["discClassL22"][i])

J15442550-2126408 nan -
J15520884-2723456 nan -
J15534211-2049282 nan -
J15551704-2322165 debris/ev-trans debris/ev-trans?
J15554314-2437435 nan -
J15572986-2258438 nan -
J15583692-2257153 nan -
J15584772-1757595 debris/ev-trans debris/ev-trans?
J15594231-2945495 nan -
J15594460-2155250 nan -
J15595759-1812234 nan -
J16001330-2418106 debris/ev-trans debris/ev-trans?
J16001844-2230114 nan -
J16011398-2516281 nan -
J16020039-2221237 debris/ev-trans debris/ev-trans?
J16020287-2236139 nan -
J16025123-2401574 debris/ev-trans debris/ev-trans?
J16041893-2430392 nan -
J16052459-1954419 debris/ev-trans debris/ev-trans?
J16052661-1957050 nan -
J16062196-1928445 nan -
J16071971-2020555 debris/ev-trans debris/ev-trans?
J16073939-1917472 debris/ev-trans debris/ev-trans?
J16082751-1949047 nan -
J16093164-2229224 nan -
J16093730-2027250 nan -
J16095441-1906551 debris/ev-trans debris/ev-trans?
J16102174-1904067 debris/ev-trans debris/ev-trans?
J16102857-1904469 nan -
J16104202-2101319 nan debris/ev-tr

`3:` Check that your 2MASS designations are the closest to the identified _Gaia_ members:

In [15]:
# Match the Luhman2022's catalogue by 2MASS ID:
matched_stars2MASS = uppersco_mine.merge(
    luhman_stars,
    left_on  = "SourceID_2MASS",
    right_on = "SourceID_2MASS_L22",
    how = "left"
)
print(len(matched_stars2MASS))

# Print number of duplicates:
dups = matched_stars2MASS[matched_stars2MASS["SourceID_2MASS"].duplicated(keep=False)]
print(len(dups))

# John did pick the closest!
for name in np.unique(dups["SourceID_2MASS"].tolist()):
    mask = matched_stars["SourceID_2MASS"] == name
    print(matched_stars["2MASS_closest"][mask])

295
22
5    Y
Name: 2MASS_closest, dtype: object
21    Y
Name: 2MASS_closest, dtype: object
23    Y
Name: 2MASS_closest, dtype: object
76    Y
Name: 2MASS_closest, dtype: object
131    Y
Name: 2MASS_closest, dtype: object
164    Y
Name: 2MASS_closest, dtype: object
194    Y
Name: 2MASS_closest, dtype: object
203    Y
Name: 2MASS_closest, dtype: object
209    Y
Name: 2MASS_closest, dtype: object
229    Y
Name: 2MASS_closest, dtype: object
249    Y
Name: 2MASS_closest, dtype: object


`4:` Check that your KP membership classification is correct.

In [16]:
# The few Ns are due to their re-evaluation by Carpenter et al. (2025) based on ALMA data.
uppersco_mine_refined = uppersco_mine[
    uppersco_mine["kinpopL22"].astype(str).str.contains("u", na=False) & 
    uppersco_mine["pospopL22"].astype(str).str.contains("u", na=False)
]
print("KP Members:", len(uppersco_mine_refined))
for i in range(len(uppersco_mine_refined)):
    print(uppersco_mine_refined["SourceID_2MASS"].tolist()[i], uppersco_mine_refined["memKP_L22"].tolist()[i])

KP Members: 177
J15482445-2235495 Y
J15514032-2146103 Y
J15530132-2114135 Y
J15540240-2254587 Y
J15562477-2225552 Y
J15564244-2039339 N
J15570146-2046184 Y
J15570641-2206060 Y
J15572109-2202130 Y
J15581270-2328364 Y
J15582981-2310077 Y
J15583620-1946135 Y
J15594426-2029232 Y
J15595116-2311044 Y
J16001330-2418106 Y
J16001730-2236504 Y
J16002945-2022536 Y
J16012268-2408003 Y
J16012652-2301343 Y
J16014086-2258103 Y
J16014157-2111380 Y
J16020039-2221237 Y
J16020429-2231468 Y
J16020517-2331070 Y
J16020757-2257467 Y
J16023587-2320170 Y
J16024152-2138245 Y
J16025123-2401574 Y
J16025855-2256495 Y
J16030161-2207523 Y
J16031329-2112569 Y
J16032225-2413111 Y
J16032277-2238206 Y
J16032625-2155378 Y
J16035228-2321076 Y
J16035767-2031055 Y
J16035793-1942108 Y
J16041416-2129151 Y
J16041740-1942287 Y
J16041792-1941505 Y
J16042165-2130284 Y
J16043916-1942459 Y
J16050231-1941554 Y
J16050844-1947070 Y
J16052076-1821367 Y
J16052157-1821412 Y
J16052459-1954419 Y
J16052556-2035397 N
J16052787-2115510 Y
J160

In [17]:
uppersco_mine_wrong = uppersco_mine[
    ~(uppersco_mine["kinpopL22"].astype(str).str.contains("u", na=False) & uppersco_mine["pospopL22"].astype(str).str.contains("u", na=False))
]
print("KP NON-Members:",len(uppersco_mine_wrong))
for i in range(len(uppersco_mine_wrong)):
    print(uppersco_mine_wrong["SourceID_2MASS"].tolist()[i], uppersco_mine_wrong["memKP_L22"].tolist()[i])

KP NON-Members: 107
J15354856-2958551 N
J15442550-2126408 -
J15462696-2443225 N
J15465432-2556520 N
J15472572-2609185 N
J15480853-2507011 N
J15485435-2443101 N
J15510126-2523100 N
J15514709-2113234 N
J15520884-2723456 -
J15521088-2125372 N
J15524851-2845369 N
J15534211-2049282 -
J15551704-2322165 N
J15554314-2437435 -
J15554883-2512240 N
J15565600-2911349 N
J15571045-2545307 N
J15572986-2258438 -
J15573049-1903014 N
J15575444-2450424 N
J15583692-2257153 -
J15584772-1757595 N
J15590484-2422469 N
J15591452-2606182 N
J15594231-2945495 -
J15594460-2155250 -
J15595759-1812234 -
J16001844-2230114 -
J16011398-2516281 -
J16012902-2509069 N
J16020287-2236139 -
J16025431-1805300 N
J16041893-2430392 -
J16044876-1748393 N
J16050647-1734020 N
J16052661-1957050 -
J16052730-2614090 N
J16052875-2655496 N
J16062196-1928445 -
J16063539-2516510 N
J16064115-2517044 N
J16071403-1702425 N
J16072863-2630130 N
J16082751-1949047 -
J16093164-2229224 -
J16093730-2027250 -
J16102857-1904469 -
J16112601-2631558 -


`5:` Import Ratzenbock's and Polnitzki's tables and check that the cluster membership and disc status are correct.
- Check "SourceID_WISE", "SourceID_GaiaDR3", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22"

In [18]:
# Load Ratzenbock2023's table:
ratzenbock = pd.read_csv(
    "tableE1_ratzenbock2023a.dat",
    header = None,
    names  = [
        "SourceID_GaiaDR3", "SigMA_membership", "Stability", "Distance (pc)", "e_d_upper (pc)", "e_d_lower (pc)",
        "X (pc)", "Y (pc)", "Z (pc)", "e_X_upper (pc)", "e_Y_upper (pc)", "e_Z_upper (pc)", "e_X_lower (pc)", "e_Y_lower (pc)", "e_Z_lower (pc)",
        "v_alpha (km s−1)", "v_delta (km s−1)", "e_v_alpha_upper (km s−1)", "e_v_delta_upper (km s−1)", "e_v_alpha_lower (km s−1)", "e_v_delta_lower (km s−1)",
        "v_alpha_LSR (km s−1)", "v_delta_LSR (km s−1)", "v_ERV (km s-1)",
    ],
    sep    = r"\s+",
)
print(len(ratzenbock))
# Numbers match (caption table E1): 13103.

13103


In [19]:
# Load Polnitky2026's table:
polnitzky = pd.read_fwf(
    "tableC2_polnitzky2026.dat",
    skiprows = 1,
    names  = [
        "SourceID_GaiaDR3", "RA (deg)", "Dec (deg)", "l (degree)", "b (degree)", "RUWE", "SourceID_AllWISE",
        "W1mag", "e_W1mag", "W2mag", "e_W2mag", "W3mag", "e_W3mag", "W4mag", "e_W4mag", "Jmag", "e_Jmag", "Hmag", "e_Hmag", "Kmag", "e_Kmag",
        "snr1", "chi2W1", "snr2", "chi2W2", "snr3", "chi2W3", "snr4", "chi2W4", 
        "Disk_CCD", "W123", "W124", "HKW2", "HKW3", "Disk_SED",
        "alpha_KW3", "e_alpha_KW3", "intercept_KW3", "e_intercept_KW3", 
        "alpha_KW4", "e_alpha_KW4", "intercept_KW4", "e_intercept_KW4", 
        "alpha_W13", "e_alpha_W13", "intercept_W13", "e_intercept_W13", 
        "alpha_W14", "e_alpha_W14", "intercept_W14", "e_intercept_W14", 
        "alpha_KW2", "e_alpha_KW2", "intercept_KW2", "e_intercept_KW2", 
        "alpha", 
        "DiskClass_Luhman", "Disk_Luhman", "SigMA_label"
    ],
)
print(len(polnitzky))
# Numbers match (direct table inspection): 10270

10270


In [20]:
# Using GaiaDR3 IDs match your table with the Ratzenbock one to double-check the kinematic membership
matched_stars = uppersco_mine.merge(
    ratzenbock[["SourceID_GaiaDR3", "SigMA_membership"]],
    on  = "SourceID_GaiaDR3",
    how = "left"
)
print(len(matched_stars))

# Using GaiaDR3 IDs match your table with the Polnitzky one to double-check the disc classification
matched_discs = uppersco_mine.merge(
    polnitzky[["SourceID_GaiaDR3", "Disk_CCD", "Disk_SED", "Disk_Luhman"]],
    on  = "SourceID_GaiaDR3",
    how = "left"
)
print(len(matched_discs))

284
284


In [21]:
# Cluster membership -> agree!
for i in range(len(matched_stars["SourceID_2MASS"])):
    if str(matched_stars["SigMACluster"][i]) != str(matched_stars["SigMA_membership"][i]): 
        print(matched_stars["SourceID_2MASS"][i], matched_stars["SigMACluster"][i], matched_stars["SigMA_membership"][i])

J15442550-2126408 - nan
J15520884-2723456 - nan
J15554314-2437435 - nan
J15572986-2258438 - nan
J15594460-2155250 - nan
J16001844-2230114 - nan
J16011398-2516281 - nan
J16020287-2236139 - nan
J16041893-2430392 - nan
J16052661-1957050 - nan
J16060061-1957114 - nan
J16062196-1928445 - nan
J16072747-2059442 - nan
J16082751-1949047 - nan
J16093164-2229224 - nan
J16093730-2027250 - nan
J16102857-1904469 - nan
J16112601-2631558 - nan
J16133650-2503473 - nan
J16135434-2320342 - nan
J16141107-2305362 - nan
J16153220-2010236 - nan
J16171889-2230017 - nan
J16181618-2619080 - nan
J16181811-2221150 - nan
J16185382-2053182 - nan
J16192288-2135037 - nan
J16204468-2431384 - nan
J16215741-2238180 - nan
J16222982-2002472 - nan
J16235468-2515392 - nan
J16251521-2511540 - nan
J16263926-2113453 - nan
J16370169-2545368 - nan
J16394272-2812141 - nan
J16413713-2730489 - nan


In [22]:
# CCD discs -> agree!
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["Disc?CCD"][i] != matched_discs["Disk_CCD"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["Disc?CCD"][i], matched_discs["Disk_CCD"][i])

J15442550-2126408 - nan
J15510126-2523100 - nan
J15520884-2723456 - nan
J15554314-2437435 - nan
J15554883-2512240 - nan
J15564244-2039339 - nan
J15570146-2046184 - nan
J15572986-2258438 - nan
J15575444-2450424 - nan
J15591452-2606182 - nan
J15594460-2155250 - nan
J16001730-2236504 - nan
J16001844-2230114 - nan
J16011398-2516281 - nan
J16012902-2509069 - nan
J16020287-2236139 - nan
J16020429-2231468 - nan
J16025123-2401574 - nan
J16041416-2129151 - nan
J16041740-1942287 - nan
J16041893-2430392 - nan
J16044876-1748393 - nan
J16052076-1821367 - nan
J16052661-1957050 - nan
J16060061-1957114 - nan
J16060215-2003142 - nan
J16061144-1935405 - nan
J16062196-1928445 - nan
J16070014-2033092 - nan
J16072682-1855239 - nan
J16072747-2059442 - nan
J16082751-1949047 - nan
J16092136-2139342 - nan
J16093164-2229224 - nan
J16093730-2027250 - nan
J16101888-2502325 - nan
J16102174-1904067 - nan
J16102857-1904469 - nan
J16104202-2101319 - nan
J16112601-2631558 - nan
J16113134-1838259 - nan
J16123352-254328

In [23]:
# SED discs -> agree!
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["Disc?SED"][i] != matched_discs["Disk_SED"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["Disc?SED"][i], matched_discs["Disk_SED"][i])

J15442550-2126408 - nan
J15510126-2523100 - nan
J15520884-2723456 - nan
J15554314-2437435 - nan
J15554883-2512240 - nan
J15564244-2039339 - nan
J15570146-2046184 - nan
J15572986-2258438 - nan
J15575444-2450424 - nan
J15591452-2606182 - nan
J15594460-2155250 - nan
J16001730-2236504 - nan
J16001844-2230114 - nan
J16011398-2516281 - nan
J16012902-2509069 - nan
J16020287-2236139 - nan
J16020429-2231468 - nan
J16025123-2401574 - nan
J16041416-2129151 - nan
J16041740-1942287 - nan
J16041893-2430392 - nan
J16044876-1748393 - nan
J16052076-1821367 - nan
J16052661-1957050 - nan
J16060061-1957114 - nan
J16060215-2003142 - nan
J16061144-1935405 - nan
J16062196-1928445 - nan
J16070014-2033092 - nan
J16072682-1855239 - nan
J16072747-2059442 - nan
J16082751-1949047 - nan
J16092136-2139342 - nan
J16093164-2229224 - nan
J16093730-2027250 - nan
J16101888-2502325 - nan
J16102174-1904067 - nan
J16102857-1904469 - nan
J16104202-2101319 - nan
J16112601-2631558 - nan
J16113134-1838259 - nan
J16123352-254328

In [24]:
# L22 discs -> agree!
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["Disc?L22"][i] != matched_discs["Disk_Luhman"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["Disc?L22"][i], matched_discs["Disk_Luhman"][i])

J15442550-2126408 - nan
J15520884-2723456 - nan
J15534211-2049282 - nan
J15554314-2437435 - nan
J15554883-2512240 - nan
J15570146-2046184 - nan
J15572986-2258438 - nan
J15575444-2450424 - nan
J15583692-2257153 - nan
J15591452-2606182 - nan
J15594231-2945495 - nan
J15594460-2155250 - nan
J15595759-1812234 - nan
J16001730-2236504 - nan
J16001844-2230114 - nan
J16011398-2516281 - nan
J16012902-2509069 - nan
J16020287-2236139 - nan
J16020429-2231468 - nan
J16025123-2401574 - nan
J16041416-2129151 - nan
J16041740-1942287 - nan
J16041893-2430392 - nan
J16044876-1748393 - nan
J16052076-1821367 - nan
J16052661-1957050 - nan
J16060061-1957114 - nan
J16060215-2003142 - nan
J16061144-1935405 - nan
J16062196-1928445 - nan
J16070014-2033092 - nan
J16072682-1855239 - nan
J16072747-2059442 - nan
J16082751-1949047 - nan
J16093164-2229224 - nan
J16093730-2027250 - nan
J16101888-2502325 - nan
J16102174-1904067 - nan
J16102857-1904469 - nan
J16104202-2101319 - nan
J16112601-2631558 - nan
J16120505-204340

`6:` Load Table D1 of Empey at al. (2026) and check that SpTs are correct.

In [25]:
# Load Empey2026's table:
empey = pd.read_csv(
    "tableD1_empey2026.dat",
    names  = ["SourceID_2MASS", "SpT_Empey", "Dist", "Teff", "AV", "Lstar", "Mstar", "Age", "logLacc", "Macc", "Mdust", "UL", "CA"],
    sep      = r"\s+",
    skiprows = 2
)
print(len(empey))
# Numbers match: 121.

121


In [26]:
matched_SpTs = uppersco_mine.merge(
    empey[["SourceID_2MASS", "SpT_Empey"]],
    on  = "SourceID_2MASS",
    how = "left"
)
print(len(matched_SpTs))
# One more because Empey includes both components of a binary (J15354856-2958551)

285


In [27]:
# SpTs agree!
for i in range(len(matched_SpTs["SourceID_2MASS"])):
    if str(matched_SpTs["SpT_Empey"][i]) != str(matched_SpTs["SpT_E26"][i]): 
        print(matched_SpTs["SourceID_2MASS"][i], matched_SpTs["SpT_Empey"][i], matched_SpTs["SpT_E26"][i])

J15462696-2443225 nan -
J15472572-2609185 nan -
J15510126-2523100 nan -
J15520884-2723456 nan -
J15524851-2845369 nan -
J15551704-2322165 nan -
J15554314-2437435 nan -
J15554883-2512240 nan -
J15564244-2039339 nan -
J15565600-2911349 nan -
J15570641-2206060 nan -
J15571045-2545307 nan -
J15572109-2202130 nan -
J15572986-2258438 nan -
J15575444-2450424 nan -
J15581270-2328364 nan -
J15584772-1757595 nan -
J15591452-2606182 nan -
J15594231-2945495 nan -
J15594460-2155250 nan -
J16001330-2418106 nan -
J16001730-2236504 nan -
J16002945-2022536 nan -
J16011398-2516281 nan -
J16012652-2301343 nan -
J16014157-2111380 nan -
J16020039-2221237 nan -
J16020287-2236139 nan -
J16020517-2331070 nan -
J16023587-2320170 nan -
J16025123-2401574 nan -
J16025855-2256495 nan -
J16030161-2207523 nan -
J16031329-2112569 nan -
J16032625-2155378 nan -
J16041416-2129151 nan -
J16041792-1941505 nan -
J16043916-1942459 nan -
J16044876-1748393 nan -
J16050231-1941554 nan -
J16050647-1734020 nan -
J16050844-194707